In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Loading
df_laeb = pd.read_csv(os.path.join(path, "labels.csv"))


imgeing_directory = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")


for index, row in df_laeb.iterrows():


    img_name = row.iloc[0]


    age = row.iloc[1]


    img_path = os.path.join(imgeing_directory, img_name)



    if os.path.exists(img_path):


        img = Image.open(img_path).convert('RGB')


        imgening_array = np.array(img) / 255.0 # Normalize  [0, 1]


        images.append(imgening_array)


        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors


import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# 1. Convert to Tensors
# useing float32 for features (X) and float32 for regression targets (y)


X_train_t = torch.tensor(X_train, dtype=torch.float32)



y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1) # Reshape to (N, 1)




X_test_t = torch.tensor(X_test, dtype=torch.float32)




y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)


In [ ]:
# 2. Creating TensorDataset object

# 2. Create Datasets


train_dataset = TensorDataset(X_train_t, y_train_t)






test_dataset = TensorDataset(X_test_t, y_test_t)


In [ ]:
# 3. Create DataLoaders

batch_size = 32


train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)


test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
# 4. Print shape of one batch

data_iter = iter(train_loader)


images_batch, labels_batch = next(data_iter)



print(f"Batch Image shape: {images_batch.shape}")  # Expected: [32, 3, H, W]


print(f"Batch Label shape: {labels_batch.shape}")  # Expected: [32, 1]


In [ ]:
# 5. Display sample images

# 5. Display a few images
def show_images(imgs, labels):
    plt.figure(figsize=(12, 5))
    for i in range(4):
        plt.subplot(1, 4, i + 1)
        # Transpose back from (C, H, W) to (H, W, C) for Matplotlib
        img = imgs[i].numpy().transpose((1, 2, 0))
        plt.imshow(img)
        plt.title(f"Age: {int(labels[i].item())}")
        plt.axis('off')
    plt.show()

show_images(images_batch, labels_batch)

In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn
import torch.optim as optim

class AgeEstimationModel(nn.Module):
    def __init__(self, input_shape):


        super(AgeEstimationModel, self).__init__()


        # Channels * Height * Width
        self.input_dim = input_shape[0] * input_shape[1] * input_shape[2]

        self.network = nn.Sequential(
            nn.Flatten(),


            nn.Linear(self.input_dim, 512),


            nn.ReLU(),


            nn.Linear(512, 256),



            nn.ReLU(),


            nn.Linear(256, 64),

            nn.ReLU(),


            nn.Linear(64, 1)  # Final output for regression
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# Task 2: Write your training loop here:


def train_step(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0


    for images, labels in loader:


        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()


        outputs = model(images)


        loss = criterion(outputs, labels)


        loss.backward()


        optimizer.step()

        running_loss += loss.item()


    return running_loss / len(loader)

def val_step(model, loader, criterion, device):


    model.eval()

    running_loss = 0.0

    with torch.no_grad():


        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
    return running_loss / len(loader)

In [ ]:
# Task 3: Write your validation loop here:
# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Initialize Model (using shape of first image in X_train)


input_shape = X_train.shape[1:]


model = AgeEstimationModel(input_shape).to(device)

# Define Loss and Optimizer


criterion = nn.MSELoss()


optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop


num_epochs = 20
history = {"training_loss": [], "valdition_loss": []}



print(f"Starting training on {device}...")
for epoch in range(num_epochs):
    train_loss = train_step(model, train_loader, criterion, optimizer, device)
    val_loss = val_step(model, test_loader, criterion, device)

    history["trainining_loss"].append(train_loss)
    history["valadtion_loss"].append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Training Loss: {train_loss:.4f}, Valdtion Loss: {val_loss:.4f}")

print("Training accoblish!")

In [ ]:
# Task 4: Define device, model, loss, optimizer:
import torch.nn as nn
import torch.optim as optim

# 1. Define Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Initialize Model
# We pass the shape of one image (C, H, W) to calculate the input dimension
input_shape = X_train.shape[1:]
model = AgeEstimationModel(input_shape).to(device)

# 3. Define Loss Function (Regression)
criterion = nn.MSELoss()

# 4. Define Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Task 5: Start training for 20 epochs:
# Storage for metrics
history = {"train_loss": [], "val_loss": []}
num_epochs = 20

print("Starting Training...")

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    train_running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        train_running_loss += loss.item() * images.size(0)

    epoch_train_loss = train_running_loss / len(train_loader.dataset)

    # --- Validation Phase ---
    model.eval()
    val_running_loss = 0.0
    with torch.no_grad(): # No need to track gradients during val
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * images.size(0)

    epoch_val_loss = val_running_loss / len(test_loader.dataset)

    # Save history
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)

    # Print progress
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

print("Training Done!")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

def plot_loss(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history['train_loss'], label='Training Loss', color='blue', lw=2)
    plt.plot(history['val_loss'], label='Validation Loss', color='orange', lw=2)
    plt.title('Training and Validation Loss Over Epochs', fontsize=14)
    plt.xlabel('Epochs')
    plt.ylabel('Mean Squared Error (MSE)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

plot_loss(history)


In [ ]:
# Task 2 (Bonus): Write your code here:
def plot_predictions(model, loader, device, num_images=5):
    model.eval()
    images, labels = next(iter(loader))
    images, labels = images.to(device), labels.to(device)

    with torch.no_grad():
        outputs = model(images)

    # Move back to CPU for plotting
    images = images.cpu()
    labels = labels.cpu()
    outputs = outputs.cpu()

    plt.figure(figsize=(15, 5))
    for i in range(num_images):
        plt.subplot(1, num_images, i + 1)

        # Reshape image: (C, H, W) -> (H, W, C)
        img = images[i].permute(1, 2, 0).numpy()

        actual_age = labels[i].item()
        pred_age = outputs[i].item()

        plt.imshow(img)
        plt.title(f"Actual: {actual_age:.1f}\nPred: {pred_age:.1f}",
                  color=("green" if abs(actual_age - pred_age) < 5 else "red"))
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# Run the visualization
plot_predictions(model, test_loader, device)